# Build a Tiny GPT, End to End, and Train It on This Blog

The full code of [Build a Tiny GPT, End to End, and Train It on This Blog](https://residual-stream.lvtan71.workers.dev/posts/tiny-gpt-from-scratch/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/tiny-gpt-from-scratch) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/tiny-gpt-from-scratch.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 314 min end to end.
Trains about twenty small models, several hours on a T4. The baseline and cosine-schedule runs reproduce the post exactly; the ablations (no position, learned positions, no residual) can differ in the third decimal.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "torch==2.14.*" tokenizers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "tiny-gpt-from-scratch"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `params.py`

Counts the model's parameters by part (embedding, attention, MLP, norms) and checks the formula 12 L D² + V D, here and for GPT-2 small.

In [ ]:
!{sys.executable} params.py

The post printed:

```text
embedding (shared with unembedding)      28,416   1.6%
attention, all blocks                   589,824  32.8%
MLP, all blocks                       1,179,648  65.5%
norm gains                                1,728   0.1%
total                                 1,799,616
formula 12 L D^2 + V D: 1,797,888  (L=4, D=192, V=148)
with post 9's 2,048 BPE tokens instead: 2,162,688, embedding 18% of it
GPT-2 small (L=12, D=768, V=50,257): 123.5M by the formula; 124M reported, the rest mostly its 1,024 learned position vectors
```

## `sweep.py`

The first round of choices on the dev slice: characters or BPE tokens, and dropout 0.1 to 0.4, one seed each. Runs `train.py` once per configuration.

In [ ]:
!{sys.executable} sweep.py

The post printed:

```text
char d0.1 s0: 250: 2.741 / 500: 2.273 / 750: 2.158 / 1000: 2.076 / 1500: 2.080 / 2000: 2.072 / 2500: 2.105 / 3000: 2.141  (46 passes)
bpe  d0.1 s0: 250: 2.415 / 500: 2.116 / 750: 2.204 / 1000: 2.396 / 1500: 2.791 / 2000: 3.066 / 2500: 3.218 / 3000: 3.333  (126 passes)
char d0.2 s0: 250: 2.919 / 500: 2.398 / 750: 2.230 / 1000: 2.147 / 1500: 2.093 / 2000: 2.053 / 2500: 2.040 / 3000: 2.034  (46 passes)
bpe  d0.2 s0: 250: 2.461 / 500: 2.129 / 750: 2.111 / 1000: 2.177 / 1500: 2.394 / 2000: 2.547 / 2500: 2.693 / 3000: 2.789  (126 passes)
char d0.2 s0: 1000: 2.147 / 2000: 2.053 / 3000: 2.034 / 4000: 2.027 / 5000: 2.029 / 6000: 2.038  (92 passes)
char d0.3 s0: 1000: 2.254 / 2000: 2.116 / 3000: 2.088 / 4000: 2.030 / 5000: 2.039 / 6000: 2.015  (92 passes)
bpe  d0.3 s0: 250: 2.522 / 500: 2.159 / 750: 2.106 / 1000: 2.114 / 1250: 2.169 / 1500: 2.201 / 2000: 2.263  (84 passes)
bpe  d0.4 s0: 250: 2.603 / 500: 2.216 / 750: 2.128 / 1000: 2.117 / 1250: 2.119 / 1500: 2.122 / 2000: 2.164  (84 passes)
```

## `ablate.py`

The second round, two seeds each on the dev slice: the chosen model, a cosine schedule, and the model without position information, with learned positions, post-norm, and without residual connections.

In [ ]:
!{sys.executable} ablate.py

The post printed:

```text
baseline        s0: 1000: 2.147 / 2000: 2.053 / 3000: 2.034 / 4000: 2.027  (61 passes)
cosine schedule s0: 1000: 2.141 / 2000: 2.016 / 3000: 2.001 / 4000: 1.990  (61 passes)
no position     s0: 1000: 3.318 / 2000: 2.954 / 3000: 2.779 / 4000: 2.706  (61 passes)
learned pos.    s0: 1000: 2.560 / 2000: 2.260 / 3000: 2.176 / 4000: 2.116  (61 passes)
post-norm       s0: 1000: 2.292 / 2000: 2.127 / 3000: 2.095 / 4000: 2.088  (61 passes)
no residual     s0: 1000: 5.061 / 2000: 5.022 / 3000: 5.049 / 4000: 5.048  (61 passes)
baseline        s1: 1000: 2.140 / 2000: 2.030 / 3000: 2.023 / 4000: 2.006  (61 passes)
cosine schedule s1: 1000: 2.134 / 2000: 2.012 / 3000: 1.992 / 4000: 1.982  (61 passes)
no position     s1: 1000: 3.374 / 2000: 2.918 / 3000: 2.790 / 4000: 2.688  (61 passes)
learned pos.    s1: 1000: 2.563 / 2000: 2.281 / 3000: 2.169 / 4000: 2.111  (61 passes)
post-norm       s1: 1000: 2.301 / 2000: 2.119 / 3000: 2.101 / 4000: 2.083  (61 passes)
no residual     s1: 1000: 5.037 / 2000: 5.051 / 3000: 5.048 / 4000: 5.043  (61 passes)
```

## `collapse.py`

The models with and without residual connections for 400 steps: training loss, the gradient at each layer's attention weights, and whether the output still depends on the input.

In [ ]:
!{sys.executable} collapse.py

The post printed:

```text
unigram entropy of the training text: 5.015 bits per character
residual=True  step   1: train loss 7.214 bits; gradient norm at the attention weights of layers 1-4: 2.0e+00 1.7e+00 1.2e+00 9.3e-01
residual=True  step 400: train loss 2.621 bits; gradient norm at the attention weights of layers 1-4: 5.6e-01 1.5e-01 1.5e-01 1.7e-01
   output distributions for two different contexts differ by 0.861 (total variation)
residual=False step   1: train loss 7.246 bits; gradient norm at the attention weights of layers 1-4: 2.0e+00 1.4e+00 1.0e+00 7.5e-01
residual=False step 400: train loss 5.086 bits; gradient norm at the attention weights of layers 1-4: 1.6e-06 1.0e-06 3.3e-06 1.1e-04
   output distributions for two different contexts differ by 0.000 (total variation)
```

## `collapse_lr.py`

The model without residual connections for 1,000 steps at three learning rates, one without dropout: does it still collapse?

In [ ]:
!{sys.executable} collapse_lr.py

The post printed:

```text
lr 1e-03, dropout 0.0: train loss 4.966 bits after 1,000 steps; two contexts' predictions differ by 0.000
lr 3e-04, dropout 0.2: train loss 4.967 bits after 1,000 steps; two contexts' predictions differ by 0.000
lr 1e-04, dropout 0.2: train loss 4.726 bits after 1,000 steps; two contexts' predictions differ by 0.294
```

## `low_lr.py`

The model without residual connections and the baseline, both at a learning rate of 1e-4, two seeds each, on the dev slice.

In [ ]:
!{sys.executable} low_lr.py

The post printed:

```text
no residual, lr 1e-4 s0: 1000: 4.400 / 2000: 3.733 / 3000: 3.398 / 4000: 3.122  (61 passes)
no residual, lr 1e-4 s1: 1000: 4.373 / 2000: 3.660 / 3000: 3.328 / 4000: 3.041  (61 passes)
baseline,    lr 1e-4 s0: 1000: 2.842 / 2000: 2.374 / 3000: 2.212 / 4000: 2.119  (61 passes)
baseline,    lr 1e-4 s1: 1000: 2.837 / 2000: 2.337 / 3000: 2.169 / 4000: 2.102  (61 passes)
```

## `score_once.py`

The scored runs: `final.py` with three seeds and the cosine schedule, trained on all the training posts and scored on the two held-out posts. Seed 0 saves `gpt_250.pt` to `gpt_4000.pt`.

In [ ]:
!{sys.executable} score_once.py

The post printed:

```text
seed 0, cosine schedule, held-out bits per byte at 250: 3.036 / 1,000: 2.511 / 2,000: 2.453 / 3,000: 2.475 / 4,000: 2.496 steps (55 passes)
seed 1, cosine schedule, held-out bits per byte at 250: 3.005 / 1,000: 2.607 / 2,000: 2.491 / 3,000: 2.562 / 4,000: 2.633 steps (55 passes)
seed 2, cosine schedule, held-out bits per byte at 250: 3.019 / 1,000: 2.492 / 2,000: 2.436 / 3,000: 2.570 / 4,000: 2.535 steps (55 passes)
```

## `lm.py char 0`



**Note:** Trains the LSTM from What "Generative" Means again and saves char_lstm.pt (a few minutes).

In [ ]:
!{sys.executable} lm.py char 0

## `per_post.py`

Each held-out post scored separately by the GPT's checkpoints, the LSTM, Kneser–Ney and bz2, plus the GPT on training text it has seen. Reads the `gpt_*.pt` checkpoints and `char_lstm.pt`.

In [ ]:
!{sys.executable} per_post.py

The post printed:

```text
123 held-out characters never occur in training: BPE post 122, KV-cache post 1
the GPT at 4,000 steps pays 25.6 bits per unseen character, 0.055 bits per byte of the held-out text in all

bits per byte                   BPE post  KV-cache post  training text (seen)
GPT, 250 steps                     3.212          2.654                 2.698
GPT, 1,000 steps                   2.775          1.935                 1.803
GPT, 2,000 steps                   2.747          1.815                 1.566
GPT, 3,000 steps                   2.801          1.763                 1.398
GPT, 4,000 steps                   2.842          1.744                 1.327
LSTM (post 9, seed 0)              2.944          2.033 
Kneser-Ney, 6 characters           2.826          2.003 
bz2 given the training posts       2.480          2.038
```

## `repeats.py`

How much of the held-out text is copied from training or repeated from beyond the model's window, and the model's cost on each kind. Reads `gpt_4000.pt`.

In [ ]:
!{sys.executable} repeats.py

The post printed:

```text
the model on the held-out posts: 2.522 bits per character
stretches of 32+ characters: copied  3.8% at 0.58 bits, repeated  1.6% at 5.15, the rest 94.7% at 2.56
stretches of 16+ characters: copied 15.4% at 1.07 bits, repeated  6.2% at 3.45, the rest 78.4% at 2.73
for comparison, the dev slice: 5.6% copied from the rest of the training text (32+)
```

## `less_data.py`

`datafrac.py` on a quarter and on half of the training text, two seeds each.

In [ ]:
!{sys.executable} less_data.py

The post printed:

```text
25% of the training text (148,404 characters), seed 0: held-out bits per byte at 250: 3.144 / 500: 2.979 / 750: 3.142 / 1,000: 3.308 / 2,000: 4.228 / 3,000: 4.878 / 4,000: 5.227 steps (221 passes)
25% of the training text (148,404 characters), seed 1: held-out bits per byte at 250: 3.142 / 500: 3.010 / 750: 3.212 / 1,000: 3.373 / 2,000: 4.178 / 3,000: 4.842 / 4,000: 5.216 steps (221 passes)
50% of the training text (296,808 characters), seed 0: held-out bits per byte at 250: 3.132 / 500: 2.869 / 750: 2.968 / 1,000: 2.916 / 2,000: 3.312 / 3,000: 3.535 / 4,000: 3.726 steps (110 passes)
50% of the training text (296,808 characters), seed 1: held-out bits per byte at 250: 3.168 / 500: 2.826 / 750: 2.915 / 1,000: 2.983 / 2,000: 3.080 / 3,000: 3.444 / 4,000: 3.608 steps (110 passes)
```

## `mfu.py`

Times training steps and reports model FLOPs utilization against the M4's measured peak of 2,878 GFLOP/s. On another GPU the times differ and the MFU line needs that GPU's own peak.

In [ ]:
!{sys.executable} mfu.py

The post printed:

```text
char: 1,799,616 parameters, 149 ms per step of 8,192 tokens, 54,915 tokens/s, 723 GFLOP/s, MFU 25.1%
bpe : 2,164,416 parameters, 161 ms per step of 8,192 tokens, 50,726 tokens/s, 778 GFLOP/s, MFU 27.0%
```

## `step_parts.py`

Times a step's parts separately: the matrix multiplies, attention's fused call with and without dropout, and AdamW's update.

In [ ]:
!{sys.executable} step_parts.py

The post printed:

```text
the 21 matrix multiplies alone, forward and backward: 45 ms, 1960 GFLOP/s
attention's fused call, 4 layers, dropout 0.0: 40 ms
attention's fused call, 4 layers, dropout 0.2: 53 ms
AdamW's update of all parameters: 1.5 ms
```

## `generate.py`

Samples from seed 0's checkpoints at 250, 1,000 and 4,000 steps, and finds the longest stretch of each sample that appears in the training posts. Reads `gpt_250.pt`, `gpt_1000.pt`, `gpt_4000.pt`.

In [ ]:
!{sys.executable} generate.py

The post printed:

```text
--- after 250 steps (temperature 0.8)
The KV cache an the rownumes the Cone
the this
***Kuad models is small in the is the methat comput, a abl.
- **The wult it's approduction the rough with recodes log-own is so
  matches the model's every over *attention the loss
blocks is every the the revery in that training, un every eform in model are math mo
[longest stretch copied from the training posts: 14 characters: "es the model's"]

--- after 1,000 steps (temperature 0.8)
The KV cache context of units to Context at it
sequence one scheduler be has is all attention 0.1. The blog loss context
process the one He in the whole recosystem to computed to the $W_V$, the
with the other attention catch from the three with the one number of $\matrix 1$.

The gradient. The model can move mo
[longest stretch copied from the training posts: 17 characters: ' from the three w']

--- after 4,000 steps (temperature 0.8)
The KV cache post
1's own exactly it can always a KV cache per layer, because it can be what
predicts above a fixed-size process that keeps the fixed prompt per log-odds
per-token context sequences and doesn't change, and dividing this post's
change in the next term training, use the serving posts are predictin
[longest stretch copied from the training posts: 19 characters: ' the serving posts ']

held-out text from character 10,000: longest stretch also in the training posts 27 characters: 's |\n| --- | --- | --- |\n| `'
held-out text from character 25,000: longest stretch also in the training posts 17 characters: '.\n\n### Where the '
held-out text from character 40,000: longest stretch also in the training posts 70 characters: ' = \\operatorname{softmax}\\!\\left(\\frac{QK^\\top}{\\sqrt{d_k}} + M\\right)'

ten more samples at 4,000 steps: longest copied stretch 15 to 38 characters
greedy, from a code prompt:
```python
import torch
from router_data import split, featurize
from overfit import sigmoid, log_loss, train

# AdamW that starts the same per-step, the same per-sequence sequence length
# structured and context problem tha
[longest stretch copied: 89 characters: 'from router_data import split, featurize\nfrom overfit import sigmoid, log_loss, train\n\n# ']
```